In [1]:
System.getProperty("java.version")

res1: String = "17.0.18"

In [2]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Spark411-Almond")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

println(s"Java: ${System.getProperty("java.version")}")
println(s"Spark: ${spark.version}")
println(s"Scala: ${scala.util.Properties.versionNumberString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 12:35:52 INFO SparkContext: Running Spark version 4.1.1
26/04/28 12:35:52 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 12:35:52 INFO SparkContext: Java version 17.0.18+8
26/04/28 12:35:52 INFO ResourceUtils: ==============================================================
26/04/28 12:35:52 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 12:35:52 INFO ResourceUtils: ==============================================================
26/04/28 12:35:52 INFO SparkContext: Submitted application: Spark411-Almond
26/04/28 12:35:52 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/28 12:35:52 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/28 12:35:52 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/28 12:35:52 INFO SecurityManager: Changing view acls to: Imp_06 - Ma26/04/28 12:35:52 INFO SecurityManager: Changing view acls to: I

2026-04-28T10:35:53.202219400Z scala-kernel-interpreter-1 ERROR An exception occurred processing Appender console org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:164)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:133)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:124)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:88)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:714)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:672)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:648)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:584)
	at org.apache.logging.log4j.

import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@53b524a9
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@3097773b

### Prueba con un Wordcount

In [3]:
val lineas = sc.parallelize(List(
  "apache spark es un motor de procesamiento distribuido",
  "spark procesa datos en memoria de forma muy eficiente",
  "scala es el lenguaje nativo de apache spark",
  "spark tiene apis en scala python java y r"
))

val palabras = lineas.flatMap(linea => linea.split(" "))
val pares = palabras.map(palabra => (palabra, 1))
val conteo = pares.reduceByKey((a, b) => a + b)
val ordenado = conteo.sortBy({ case (_, count) => count }, ascending = false)

val top10 = ordenado.take(10)

println("Top 10 palabras más frecuentes:")
println("─" * 35)
top10.foreach { case (palabra, count) =>
  println(f"  $palabra%-30s → $count veces")
}

Top 10 palabras más frecuentes:
───────────────────────────────────
  spark                          → 4 veces
  de                             → 3 veces
  en                             → 2 veces
  scala                          → 2 veces
  es                             → 2 veces
  apache                         → 2 veces
  eficiente                      → 1 veces
  memoria                        → 1 veces
  nativo                         → 1 veces
  y                              → 1 veces


lineas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd3.sc:1
palabras: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[1] at flatMap at cmd3.sc:8
pares: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[2] at map at cmd3.sc:9
conteo: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[3] at reduceByKey at cmd3.sc:10
ordenado: org.apache.spark.rdd.RDD[(String, Int)] = MapPartitionsRDD[8] at sortBy at cmd3.sc:11
top10: Array[(String, Int)] = Array(
  ("spark", 4),
  ("de", 3),
  ("en", 2),
  ("scala", 2),
  ("es", 2),
  ("apache", 2),
  ("eficiente", 1),
  ("memoria", 1),
  ("nativo", 1),
  ("y", 1)
)

### Ejemplo ampliado de: combineByKey

Uso de una de las funciones más avanzadas y potentes de Spark: `combineByKey`.

Mientras que `reduceByKey` se usa para operaciones simples (como sumar), `combineByKey` te permite cambiar el tipo de los datos (en este caso, pasas de valores Double individuales a una List[Double] acumulada).

In [4]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("PairRDDs")
  .master("local[*]")
  .getOrCreate()

val sc = spark.sparkContext

// RDD de ventas como cadenas: "producto,región,importe"
val ventas = sc.parallelize(List(
  "Laptop,Norte,1200",
  "Teclado,Sur,45",
  "Monitor,Norte,350",
  "Laptop,Sur,1200",
  "Ratón,Norte,25",
  "Monitor,Sur,350",
  "Teclado,Norte,45"
))

// Convertimos a Pair RDD: (región, importe)
val ventasPorRegion = ventas.map { linea =>
  val campos = linea.split(",")
  val region  = campos(1)
  val importe = campos(2).toDouble
  (region, importe)   // ← tupla (clave, valor)
}

ventasPorRegion.collect().foreach(println)
// (Norte,1200.0)
// (Sur,45.0)
// (Norte,350.0)
// (Sur,1200.0)
// (Norte,25.0)
// (Sur,350.0)
// (Norte,45.0)

// Objetivo: para cada región → List con todos los importes
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),                       // createCombiner: primer valor → List
  (acc: List[Double], v: Double) => acc :+ v,   // mergeValue: añadir al List
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2 // mergeCombiners: unir Lists 
)

importesPorRegion.collect().foreach { case (region, lista) =>
  println(s"$region → $lista")
}
// Norte → List(1200.0, 350.0, 25.0, 45.0)
// Sur   → List(45.0, 1200.0, 350.0)

(Norte,1200.0)
(Sur,45.0)
(Norte,350.0)
(Sur,1200.0)
(Norte,25.0)
(Sur,350.0)
(Norte,45.0)
Sur → List(45.0, 1200.0, 350.0)
Norte → List(1200.0, 350.0, 25.0, 45.0)


import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@53b524a9
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@3097773b
ventas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[9] at parallelize at cmd4.sc:12
ventasPorRegion: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[10] at map at cmd4.sc:23
importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[11] at combineByKey at cmd4.sc:43

##### Mas Detalle: Ejecución completa

In [5]:
val importesPorRegion = ventasPorRegion.combineByKey(
  (v: Double) => List(v),
  (acc: List[Double], v: Double) => acc :+ v,
  (acc1: List[Double], acc2: List[Double]) => acc1 ++ acc2
)

importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[12] at combineByKey at cmd5.sc:4

##### Mas Detalle: Versión con prints

In [6]:
val importesPorRegion = ventasPorRegion.combineByKey(

  (v: Double) => {
    println(s"[CREATE] Nuevo acumulador con: $v")
    List(v)
  },

  (acc: List[Double], v: Double) => {
    println(s"[MERGE VALUE] Lista actual: $acc | Añadiendo: $v")
    acc :+ v
  },

  (acc1: List[Double], acc2: List[Double]) => {
    println(s"[MERGE COMBINERS] Uniendo: $acc1 y $acc2")
    acc1 ++ acc2
  }

)

importesPorRegion.collect().foreach(println)

[CREATE] Nuevo acumulador con: 1200.0
[CREATE] Nuevo acumulador con: 350.0
[CREATE] Nuevo acumulador con: 1200.0
[CREATE] Nuevo acumulador con: 25.0
[CREATE] Nuevo acumulador con: 350.0
[CREATE] Nuevo acumulador con: 45.0
[CREATE] Nuevo acumulador con: 45.0
[MERGE COMBINERS] Uniendo: List(45.0) y List(1200.0)
[MERGE COMBINERS] Uniendo: List(1200.0) y List(350.0)
[MERGE COMBINERS] Uniendo: List(1200.0, 350.0) y List(25.0)
[MERGE COMBINERS] Uniendo: List(45.0, 1200.0) y List(350.0)
[MERGE COMBINERS] Uniendo: List(1200.0, 350.0, 25.0) y List(45.0)
(Sur,List(45.0, 1200.0, 350.0))
(Norte,List(1200.0, 350.0, 25.0, 45.0))


importesPorRegion: org.apache.spark.rdd.RDD[(String, List[Double])] = ShuffledRDD[13] at combineByKey at cmd6.sc:13

El código anterior define el viaje de transformación de los datos de ventas. Tu nueva función verParticiones es la herramienta de auditoría para ver si ese viaje está bien equilibrado entre todos los recursos de tu máquina (ese local[*] que pusiste en el master).

In [7]:
// 1. Primero defines la herramienta (lo que ya tienes)
def verParticiones[T](rdd: org.apache.spark.rdd.RDD[T]): Unit = {
  rdd.mapPartitionsWithIndex { (index, iter) =>
    Iterator(s"📦 Partición $index -> ${iter.toList.mkString(", ")}")
  }.collect().foreach(println)
}

// 2. AHORA la usas con tus RDDs para ver la magia
println("Distribución del RDD original (ventas):")
verParticiones(ventas)

println("\nDistribución del Pair RDD (ventasPorRegion):")
verParticiones(ventasPorRegion)


Distribución del RDD original (ventas):
📦 Partición 0 -> 
📦 Partición 1 -> Laptop,Norte,1200
📦 Partición 2 -> Teclado,Sur,45
📦 Partición 3 -> Monitor,Norte,350
📦 Partición 4 -> Laptop,Sur,1200
📦 Partición 5 -> Ratón,Norte,25
📦 Partición 6 -> Monitor,Sur,350
📦 Partición 7 -> Teclado,Norte,45

Distribución del Pair RDD (ventasPorRegion):
📦 Partición 0 -> 
📦 Partición 1 -> (Norte,1200.0)
📦 Partición 2 -> (Sur,45.0)
📦 Partición 3 -> (Norte,350.0)
📦 Partición 4 -> (Sur,1200.0)
📦 Partición 5 -> (Norte,25.0)
📦 Partición 6 -> (Sur,350.0)
📦 Partición 7 -> (Norte,45.0)


defined function verParticiones